In [ ]:
import sys, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "modules" / "RNA-FM"))
sys.path.insert(0, str(REPO_ROOT / "modules" / "pipeline"))
sys.path.insert(0, str(REPO_ROOT / "modules" / "global_PCA"))

import short_ncrna as sn
from pyrion import TwoBitAccessor
from pyrion.io.bed import read_bed12_file

SEED = 42
random.seed(SEED); np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

## Load model + RNA-FM token embedding helper

`emb_fn(seq)` returns `(L, 640)` token embeddings (BOS/EOS stripped).

In [ ]:
import fm

fm_model, fm_alpha = fm.pretrained.rna_fm_t12()
fm_model.eval().to(device)
fm_bc = fm_alpha.get_batch_converter()
EMB_DIM = 640

def emb_fn(seq: str) -> np.ndarray:
    rna = seq.upper().replace("T", "U")
    _, _, tk = fm_bc([("x", rna)])
    tk = tk.to(device)
    with torch.no_grad():
        out = fm_model(tk, repr_layers=[12])["representations"][12]
    return out[0, 1:1 + len(rna), :].cpu().float().numpy()

print(f"RNA-FM ready, emb dim = {EMB_DIM}")

## Load sequences

Panel-B targets (same as `create_figure_1.ipynb`) plus a small pool of ncRNAs to fit
a **global PCA** on. Pool ≈ 50 tRNA + 50 snoRNA + 50 miRNA, capped at 300 nt.

In [ ]:
BED_PATH    = REPO_ROOT / "input_data" / "reference_annotation" / "hg38.input.w.tRNA.bed"
TWOBIT_PATH = REPO_ROOT / "input_data" / "2bit" / "hg38.2bit"
META_PATH   = REPO_ROOT / "input_data" / "reference_annotation" / "test_sample.metadata.tsv"

accessor = TwoBitAccessor(str(TWOBIT_PATH))
bed_data = read_bed12_file(str(BED_PATH))
meta_df  = pd.read_csv(META_PATH, sep="	")
biotype  = dict(zip(meta_df["transcript_id"], meta_df["transcript_biotype"]))

PANEL_B = {
    "tRNA-Asn-GTT-chr1-140": {"color": "#1a3a5c", "label": "tRNA-Asn (copy 1)"},
    "tRNA-Asn-GTT-chr1-139": {"color": "#6baed6", "label": "tRNA-Asn (copy 2)"},
    "ENST00000362168.1":     {"color": "#e8720c", "label": "miRNA (MIR103A1)"},
}

# Panel-B sequences
panel_b_seqs = {}
for t in bed_data:
    if t.id in PANEL_B:
        s = sn._get_spliced_sequence(t, accessor)
        panel_b_seqs[t.id] = s.upper().replace("T", "U")
for k, v in panel_b_seqs.items():
    print(f"  {PANEL_B[k]['label']}: {len(v)} nt")

# Pool for global PCA fitting
WANTED = {"tRNA": 50, "snoRNA": 50, "miRNA": 50}
buckets = {b: [] for b in WANTED}
for t in bed_data:
    if t.id.startswith("tRNA-") and "Und-NNN" not in t.id:
        buckets["tRNA"].append(t)
    else:
        b = biotype.get(t.id.split(".")[0])
        if b in WANTED:
            buckets[b].append(t)

pool_seqs = []
rng_pool = random.Random(SEED)
for b, lst in buckets.items():
    rng_pool.shuffle(lst)
    n_taken = 0
    for t in lst:
        if n_taken >= WANTED[b]:
            break
        s = sn._get_spliced_sequence(t, accessor)
        if s and "N" not in s.upper() and 20 <= len(s) <= 300:
            pool_seqs.append(s.upper().replace("T", "U"))
            n_taken += 1
    print(f"  pool {b}: {n_taken}")

print(f"
Pool total: {len(pool_seqs)} sequences")

## Fit a global PCA on the pool

We embed every pool sequence, stack all per-position vectors, fit PCA, and keep the
smallest k that reaches ~85% explained variance (same target as `rnafm_pca_k16.npz`).

In [ ]:
VAR_TARGET = 0.85
MAX_K = 64

print("Embedding pool...")
chunks = []
for i, s in enumerate(pool_seqs):
    chunks.append(emb_fn(s))
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(pool_seqs)}")
X_pool = np.vstack(chunks).astype(np.float32)
print(f"Pool token matrix: {X_pool.shape}")

pca_full = PCA(n_components=min(MAX_K, X_pool.shape[1])).fit(X_pool)
k_keep = int(np.searchsorted(np.cumsum(pca_full.explained_variance_ratio_), VAR_TARGET) + 1)
global_pca = PCA(n_components=k_keep).fit(X_pool)

print(f"Keeping k = {k_keep} components ({global_pca.explained_variance_ratio_.sum():.1%} variance)")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(np.arange(1, len(pca_full.explained_variance_ratio_) + 1),
        np.cumsum(pca_full.explained_variance_ratio_), marker="o", ms=3)
ax.axhline(VAR_TARGET, color="red", lw=0.7, ls="--")
ax.axvline(k_keep, color="grey", lw=0.7, ls=":")
ax.set_xlabel("Components"); ax.set_ylabel("Cumulative variance")
ax.set_title(f"RNA-FM global PCA on pool ({len(pool_seqs)} seqs, {X_pool.shape[0]} tokens)")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.show()

## Test 1 — Panel-B MMD: raw vs global-PCA space

Expect `MMD(tRNA copy 1, tRNA copy 2) ≪ MMD(tRNA, miRNA)` in both spaces.

In [ ]:
raw_embs = {tid: emb_fn(s) for tid, s in panel_b_seqs.items()}
pca_embs = {tid: global_pca.transform(v) for tid, v in raw_embs.items()}

rows = []
keys = list(panel_b_seqs.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        rows.append({
            "pair":    f"{PANEL_B[keys[i]]['label']}  vs  {PANEL_B[keys[j]]['label']}",
            "mmd_raw": sn._compute_mmd(raw_embs[keys[i]], raw_embs[keys[j]]),
            "mmd_pca": sn._compute_mmd(pca_embs[keys[i]], pca_embs[keys[j]]),
        })
panel_b_df = pd.DataFrame(rows)
print(panel_b_df.to_string(index=False))

# PC1/PC2 scatter for visual sanity
fig, ax = plt.subplots(figsize=(5, 5))
for tid, meta in PANEL_B.items():
    p = pca_embs[tid]
    ax.scatter(p[:, 0], p[:, 1], c=meta["color"], label=meta["label"],
               s=22, alpha=0.75, linewidths=0, zorder=3)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.set_title("RNA-FM per-nt embeddings in global PCA space")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

## Test 2 + 3 — perturbation sensitivity & random baselines

Reference: `tRNA-Asn (copy 1)`.

- **Perturbations**: substitute a fraction `f ∈ {1, 2, 5, 10, 20}%` of bases with a different base. 10 repeats per f.
- **Shuffled**: full permutation of the reference (preserves composition). 20 repeats.
- **Random**: i.i.d. AUGC same length as reference. 20 repeats.

All MMD computed in the **global PCA space** (matching how the pipeline runs).

In [ ]:
BASES = "AUGC"

def point_mutate(seq, frac, rng):
    s = list(seq)
    n = max(1, int(round(frac * len(s))))
    for i in rng.sample(range(len(s)), n):
        s[i] = rng.choice([b for b in BASES if b != s[i]])
    return "".join(s)

def shuffled(seq, rng):
    s = list(seq); rng.shuffle(s); return "".join(s)

def random_seq(length, rng):
    return "".join(rng.choices(BASES, k=length))

def mmd_pca(a_seq, b_seq):
    return sn._compute_mmd(global_pca.transform(emb_fn(a_seq)),
                           global_pca.transform(emb_fn(b_seq)))

REF_ID  = "tRNA-Asn-GTT-chr1-140"
ref_seq = panel_b_seqs[REF_ID]
FRACS   = [0.01, 0.02, 0.05, 0.10, 0.20]
N_REP   = 10
N_NULL  = 20

rng = random.Random(SEED)

records = []
for f in FRACS:
    for _ in range(N_REP):
        mut = point_mutate(ref_seq, f, rng)
        records.append({"kind": f"mut_{int(f*100)}%", "mmd": mmd_pca(ref_seq, mut)})
    print(f"  done mut={int(f*100)}%")

for _ in range(N_NULL):
    records.append({"kind": "shuffled", "mmd": mmd_pca(ref_seq, shuffled(ref_seq, rng))})
for _ in range(N_NULL):
    records.append({"kind": "random",   "mmd": mmd_pca(ref_seq, random_seq(len(ref_seq), rng))})

df = pd.DataFrame(records)
summary = df.groupby("kind")["mmd"].agg(["mean", "std", "min", "max", "count"]).reset_index()
print()
print(summary.to_string(index=False))

In [ ]:
order = [f"mut_{int(f*100)}%" for f in FRACS] + ["shuffled", "random"]
colors = ["#1b9e77"] * len(FRACS) + ["#d95f02", "#7570b3"]

fig, ax = plt.subplots(figsize=(8, 4.5))
data = [df[df.kind == k]["mmd"].values for k in order]
bp = ax.boxplot(data, labels=order, showfliers=False, patch_artist=True, widths=0.55)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c); patch.set_alpha(0.5); patch.set_edgecolor("black"); patch.set_linewidth(0.7)

# overlay the between-class MMD from Panel B as reference
between_class = panel_b_df.loc[panel_b_df["pair"].str.contains("miRNA"), "mmd_pca"].mean()
ax.axhline(between_class, color="black", lw=0.7, ls="--",
           label=f"tRNA vs miRNA MMD (PCA) = {between_class:.3f}")

ax.set_ylabel("MMD vs reference tRNA-Asn (PCA space)")
ax.set_title("RNA-FM: perturbation sensitivity vs random baseline")
ax.tick_params(axis="x", rotation=30)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()